<blank>

# **SFRUTTO LE CLASSI DI SCIKIT-LEARN**

## **SVM PCA**

## Preparazione dei dati

---

In [28]:
import numpy as np
# Segnale su cui effettuare Train e Validation
g_0_b = np.loadtxt('data/g_0_signal_b.txt')
g_1_b = np.loadtxt('data/g_1_signal_b.txt')
# Segnale su cui effettuare Test Esterno
g_0_a = np.loadtxt('data/g_0_signal_a.txt')
g_1_a = np.loadtxt('data/g_1_signal_a.txt')

Concatenazione gruppi e vettore dei label:

In [29]:
N0, N1 = g_0_b.shape[0], g_1_b.shape[0]
# concateno g_0 e g_1
signal_b = np.vstack((g_0_b, g_1_b))
signal_a = np.vstack((g_0_a, g_1_a))

# vettore delle risposte
labels = np.concatenate((np.zeros(N0), np.ones(N1))) # uguale per A e B

<blank>

## Train + Validation con `GridSearchCV`

---

### Premessa

Con l'oggetto `Pipeline` dividiamo in 3 step il processo, due di pre-processing del segnale, una di effettivo training del modello:
1. `"scaling"`

    Proviamo diversi metodi di scaling:
    - `StandardScaler`: $$X'_\text{Standard}=\frac{X-\mu}{\sigma}$$
    - `MinMaxScaler`: $$X'_\text{MinMax}=\frac{X-X_\text{min}}{X_\text{max}-X_\text{min}}$$
    - `RobustScaler`: $$X'_\text{Robust}=\frac{X-X_\text{mediana}}{IQR}$$

1. `"reduce_dim"`

    Riduzione delle componenti tramite `PCA`; proviamo diversi _numeri di componenti_. 

1. `"classify"`

    Classificazione tramite **Support Vector Machine** (`svm.SVC`); proviamo diversi _kernel_.



`param_grid` è il dizionario che contiene le opzioni dei parametri da fornire a `GridSearchCV`. La sintassi è particolare: con `"__"` si può fare riferimento ad un argomento/membro di una classe fornita in `Pipeline` (e.g. `"nome_step__parametro": valori_da_provare`).

La strategia di cross-validation scelta è `RepeatedStratifiedKFold`:
- **'Stratified'** perché non divide i sample in `n_splits` blocchi statici, ma li costruisce in modo randomico; ci si aspetta maggiore bilanciamento nel numero di positivi e negativi in ciascun fold;
- **'Repeated'** perché ripete per `n_repeats` volte lo split in modi diversi, per diminuire la dipendenza del risultato dal rimescolamento dei dati scelto (i.e. dal seed).

<p align="center">
  <img src="images/StratifiedKFold-scheme.png" alt="Schema k-fold cross-validation">
</p>

`GridSearchCV.fit()` calcolerà l'accuracy $s$ con la strategia di cross-validation scelta su _ciascun split_, per _ciascuna combinazione_ di parametri forniti in `param_grid`. La migliore combinazione di parametri, almeno secondo chi ha scritto questo oggetto, è quella che dà lo _score_ medio $\langle s \rangle$ migliore. $\langle s \rangle$ è calcolato mediando tutti gli $s$ misurati, nel nostro caso sono `n_splits * n_repeats = 5 * 20 = 100`.

Nel dizionario `GridSearchCV.cv_results_` vengono raccolti tutti i risultati del processo di ottimizzazione, per ciascuna combinazione di parametri sono salvati TUTTI gli _score_ ottenuti.

### Codice

In [ ]:
from sklearn.decomposition import PCA
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler
from sklearn import svm

import pickle as pkl
import pandas as pd
import os

# Se il risultato esiste già, non ricalcolare tutto
results_path = "results/risultati_GridSearch_df.pkl"
model_path = "results/miglior_modello.pkl"

if os.path.exists(results_path) and os.path.exists(model_path):
    # Carichiamo il DataFrame
    results_df = pd.read_pickle(results_path)
    # Carichiamo l'oggetto del modello (la Pipeline vincente)
    with open(model_path, 'rb') as f:
        best_model = pkl.load(f)

else:

    rkf = RepeatedStratifiedKFold(n_splits=5, n_repeats=20, random_state=42)

    N_COMPONENTS_OPTIONS = [2, 5, 10, 15, 20]
    C_OPTIONS = [0.00001, 0.0001, 0.001, 0.1, 1, 10]
    KERNEL_OPTIONS = ["linear", "poly"] # vince sempre linear ["poly", "rbf", "sigmoid"]

    # 1. Definizione Pipeline #
    pipe = Pipeline([
        # Step 1: Scaling (# NOTE: va inserito uno scaler placeholder)
        ("scaling", StandardScaler()),       
        
        # Step 2: Riduzione dimensionalità (PCA)
        ("reduce_dim", PCA(random_state=42)),
        
        # Step 3: Classificatore
        ("classify", svm.SVC(random_state=42)) 
    ])

    # 2. Definizione griglia dei parametri #
    param_grid = {
        # Per provare diversi oggetti Scaler
        "scaling": [StandardScaler(), MinMaxScaler(), RobustScaler()],
        
        # Per provare parametri specifici di uno step (PCA)
        "reduce_dim__n_components": N_COMPONENTS_OPTIONS, 
        
        # Per provare parametri del classificatore
        "classify__C": C_OPTIONS,
        
        # Per provare diversi kernel
        "classify__kernel": KERNEL_OPTIONS,
    }

    # 3. Configurazione GridSearch #
    grid = GridSearchCV(
        pipe, 
        param_grid=param_grid, 
        cv=rkf,
        n_jobs=-1, # «Number of jobs to run in parallel. -1 means using all processors»
        scoring='accuracy'
    )

    # 4. Training e Validation (su Segnale B) #
    grid.fit(signal_b, labels)

    # 5. Risultati #
    print(f"La miglior configurazione: {grid.best_params_}")
    print(f"Fornisce accuracy in validation: {grid.best_score_:.4f}")
    
    # Conversione dei risultati in DataFrame
    results_df = pd.DataFrame(grid.cv_results_)

    # HACK: Per analizzare in un secondo momento
    # Salvataggio dei risultati in un file pickle
    results_df.to_pickle(results_path)
    
    with open(model_path, 'wb') as f:
        pkl.dump(grid.best_estimator_, f)

La miglior configurazione: {'classify__C': 0.1, 'classify__kernel': 'linear', 'reduce_dim__n_components': 2, 'scaling': MinMaxScaler()}
Fornisce accuracy in validation: 0.6863


> 🟩 **Dopo un test da 22 minuti**\
La miglior configurazione: {'classify__C': 0.1, 'classify__kernel': 'linear', 'reduce_dim__n_components': 2, 'scaling': MinMaxScaler()}\
Fornisce accuracy in validation: 0.6863

<blank>

## Analisi della fase di validation

---

### Classifica degli estimator

In [45]:
# Ciascuna combinazione di parametri è una riga
print(f"Numero totale di configurazioni provate: {results_df.shape[0]}")

# Selezioniamo solo le colonne interessanti per pulire la vista
columns_to_show = [
    'param_scaling', 
    'param_reduce_dim__n_components', 
    'param_classify__C', 
    'param_classify__kernel',
    'mean_test_score', 
    'std_test_score', 
    'rank_test_score'
]

# Ordiniamo per classifica (rank_test_score)
analysis = results_df[columns_to_show].sort_values('rank_test_score')

# Se si ha, è comodo aprire analysis in un viewer tipo Data Wrangler
analysis.head(20)

Numero totale di configurazioni provate: 180


,param_scaling,param_reduce_dim__n_components,param_classify__C,param_classify__kernel,mean_test_score,std_test_score,rank_test_score
91,MinMaxScaler(),2,0.1000,linear,0.686250,0.151243,1
121,MinMaxScaler(),2,1.0000,linear,0.685000,0.148096,2
151,MinMaxScaler(),2,10.0000,linear,0.685000,0.148096,2
90,StandardScaler(),2,0.1000,linear,0.684861,0.152888,4
120,StandardScaler(),2,1.0000,linear,0.684861,0.152888,4
150,StandardScaler(),2,10.0000,linear,0.684861,0.152888,4
94,MinMaxScaler(),5,0.1000,linear,0.680417,0.141333,7
60,StandardScaler(),2,0.0010,linear,0.677778,0.153081,8
63,StandardScaler(),5,0.0010,linear,0.675000,0.150449,9
154,MinMaxScaler(),5,10.0000,linear,0.672778,0.145261,10


> 💬 **Commento qualitativo**: \
`MinMaxScaler` e `StandardScaler` per tre valori di `C` restituiscono risultati praticamente identici, i parametri più determinanti sembrano quindi essere il `kernel` (linear) e il numero di componenti della `PCA` (2).

### Analisi quantitativa (⚠️ ancora da fare)

In [ ]:
# TODO

## Analisi della riduzione dei parametri (⚠️ ancora da fare)

---

Autovalori più importanti? Quanti? 

In [ ]:
# TODO

<blank>

## Test

---

### Score del miglior modello (2 componenti)

Si testa sui dati esterni (segnale A) col metodo `GridSearchCV.score()`, che utilizza lo stimatore migliore (i.e. i migliori parametri).

In [46]:
accuracy_finale = grid.score(signal_a, labels)
print(f"Risultato sul set indipendente (Segnale A): {accuracy_finale:.4f}")

Risultato sul set indipendente (Segnale A): 0.3864


### Score del modello settimo classificato (5 componenti)

Il risultato è terribile, peggio del lancio di una monetina. Il sospetto è che il modello 'migliore' sia fin troppo trainato sui dati di B. Un possibile colpevole sono le 2 sole dimensioni nello spazio delle componenti: provo a prendere il primo estimator con più di 2 componenti: è il settimo in classifica e ha parametri: 

In [57]:
# con .iloc prendi la settima posizione, con .loc prenderesti l'elemento con 'nome' 6
analysis.iloc[6]

param_scaling                     MinMaxScaler()
param_reduce_dim__n_components                 5
param_classify__C                            0.1
param_classify__kernel                    linear
mean_test_score                         0.680417
std_test_score                          0.141333
rank_test_score                                7
Name: 94, dtype: object

Definisco una nuova `Pipeline` per effettuare lo stesso pre-processing dei dati con `MinMaxScaler` e `PCA`, stavolta però non serve `GridSearchCV`: abbiamo dei parametri da fissare!

In [66]:
alternative_pipe = Pipeline([
    ("scaling", MinMaxScaler()),       
    ("reduce_dim", PCA(n_components=5, random_state=42)),
    ("classify", svm.SVC(C=0.1, kernel="linear", random_state=42)) 
])

alternative_pipe.fit(signal_b, labels)
alt_accuracy = alternative_pipe.score(signal_a, labels)

print(f"L'accuracy del settimo modello è: {alt_accuracy:.4f}")

L'accuracy del settimo modello è: 0.4773


Questo si avvicina decisamente di più al 50%.

### Score dei migliori 10 modelli

Provo anche gli altri 8 della top ten. Mi aspetto di vedere che i primi 6 sono tutti come il primo: troppo bravi sui dati di training e quindi terribili su quelli di test.

In [67]:
print("Accuracy dei modelli in top 10 sul segnale A")
for i in range(10):
    # il kernel migliore è sempre linear
    scaler = analysis.iloc[i]["param_scaling"]
    n_comp = analysis.iloc[i]["param_reduce_dim__n_components"]
    C_val  = analysis.iloc[i]["param_classify__C"]
    
    top_ten_pipe = Pipeline([
    ("scaling", scaler),       
    ("reduce_dim", PCA(n_components=n_comp, random_state=42)),
    ("classify", svm.SVC(C=C_val, kernel="linear", random_state=42)) 
    ])

    top_ten_pipe.fit(signal_b, labels)
    acc = top_ten_pipe.score(signal_a, labels)

    print(f"{i+1}. acc = {acc:.4f}")


Accuracy dei modelli in top 10 sul segnale A
1. acc = 0.3864
2. acc = 0.4091
3. acc = 0.4091
4. acc = 0.4091
5. acc = 0.4091
6. acc = 0.4091
7. acc = 0.4773
8. acc = 0.3864
9. acc = 0.4545
10. acc = 0.4545


> 💬 **Commento qualitativo**: \
Il migliore sulla predizione pare proprio essere il settimo classificato, ovvero il miglior modello con più di 2 componenti sopravvissute alla `PCA`.